# All-gage mean-flow fit test for targeted MOI basins

This notebook implements Mike's proposed in-sample capacity test:

1. Read the existing continental SOS validation products.
2. Keep basins `7429`, `7426`, and `6424`, then select three additional gage-rich basins with the worst final-MOI median `|nBias|`.
3. Make an experiment-only Cal/Val CSV in which every listed gage in those six basins is calibration.
4. Run constrained MOI directly against the existing Unity `input/` and `flpe/` directories.
5. Export the exact observed and fitted mean-flow values held by the solver, then create residual tables and ECDFs.

This is an **in-sample fit-capacity diagnostic**, not an independent validation experiment. The original Cal/Val CSV and the existing production outputs are never modified.

In [ ]:
# Cell 1 — configuration
from pathlib import Path
import importlib
import os
import sys
import warnings

RUN_ROOT = Path(
    "/nas/cee-water/cjgleason/Yushan/Confluence_Aug/"
    "confluence_global_v17c_gagecorr/global_v17c_gagecorr_mnt"
)
RUN_TAG = "all_gage_mean_fit_v1"
RESULT_ROOT = RUN_ROOT / "experiments" / RUN_TAG

FIXED_BASINS = ["7429", "7426", "6424"]
N_EXTRA_BASINS = 3
MIN_ALL_GAGES = 10
MIN_VALIDATION_GAGES = 3
RANK_METRIC = "median_abs_nbias_moi"
BASIN_ID_DIGITS = 4

# Running six basins can take a while. Existing per-basin CSVs are reused
# after an interrupted kernel. Change RUN_TAG when changing model settings.
RUN_MOI_EXPERIMENT = True
REUSE_EXISTING = True
CONTINUE_ON_ERROR = True
WRITE_REACH_NETCDF = False  # True writes every reach NetCDF; not needed for the residual test.
VERBOSE = True

# Empty means use the current set_moi_params() values from this checkout.
# Example sensitivity test: {"Gage_Uncertainty": 0.01}
PARAMETER_OVERRIDES = {}

# If cvxpy or its solvers are absent from the active Jupyter kernel, Cell 2
# installs the missing MOI solve stack into that kernel's environment.
AUTO_INSTALL_MISSING_MOI_PACKAGES = True

# Normally the notebook finds the repository automatically. Set this to an
# explicit Path only if the notebook is copied outside the repository.
MOI_REPO_OVERRIDE = None

cwd = Path.cwd().resolve()
repo_candidates = [
    MOI_REPO_OVERRIDE,
    cwd,
    cwd.parent,
    Path("/nas/cee-water/cjgleason/Yushan/Confluence_Aug/MOI_gage_correlation"),
]
MOI_REPO = next(
    (Path(path).resolve() for path in repo_candidates if path is not None and (Path(path) / "run_MOI.py").is_file()),
    None,
)
if MOI_REPO is None:
    raise FileNotFoundError(
        "Cannot locate the MOI repository. Set MOI_REPO_OVERRIDE in Cell 1."
    )
ANALYSIS_DIR = MOI_REPO / "analysis"
if str(ANALYSIS_DIR) not in sys.path:
    sys.path.insert(0, str(ANALYSIS_DIR))

from all_gage_mean_fit import (
    build_basin_ranking,
    discover_basin_catalog,
    discover_svs_file,
    load_moi_validation_samples,
    plot_residual_ecdfs,
    read_calval_table,
    run_selected_basins,
    summarize_residuals,
    write_all_gage_calval_csv,
)

SOURCE_CALVAL_CSV = MOI_REPO / "CalValSeparation_basin_stratified_v2.csv"
SOS_RESULT_DIR = RUN_ROOT / "output" / "sos"
SVS_DIR = RUN_ROOT / "input" / "svs"
ALL_GAGE_CALVAL_CSV = RESULT_ROOT / "config" / "CalVal_all_gages_selected_basins.csv"

print(f"MOI repository : {MOI_REPO}")
print(f"Unity run root : {RUN_ROOT}")
print(f"Result root    : {RESULT_ROOT}")
print(f"SLURM job ID   : {os.environ.get('SLURM_JOB_ID', 'not set')}")

In [ ]:
# Cell 2 — prepare and validate the complete MOI runtime
import site
import subprocess

required_packages = [
    "numpy", "pandas", "scipy", "netCDF4", "matplotlib",
    "cvxpy", "osqp", "scs",
]

def find_missing_packages():
    missing = []
    for package in required_packages:
        try:
            importlib.import_module(package)
        except ImportError:
            missing.append(package)
    return missing

missing_packages = find_missing_packages()
if missing_packages and AUTO_INSTALL_MISSING_MOI_PACKAGES:
    # Install only the solve stack here. Core scientific/NetCDF packages
    # should come from the Unity MOI environment to avoid replacing NumPy
    # underneath an already-running kernel.
    solver_stack = ["cvxpy", "osqp", "scs", "clarabel"]
    packages_to_install = [
        package for package in solver_stack
        if package in missing_packages or package == "clarabel"
    ]
    if packages_to_install:
        site_directories = [
            Path(path) for path in site.getsitepackages() if Path(path).exists()
        ]
        site_is_writable = any(os.access(path, os.W_OK) for path in site_directories)
        pip_command = [sys.executable, "-m", "pip", "install"]
        if not site_is_writable:
            if not site.ENABLE_USER_SITE:
                raise PermissionError(
                    "The kernel environment is read-only and user-site packages are disabled. "
                    "Create/register the MOI virtual environment shown below this cell."
                )
            pip_command.append("--user")
        pip_command.extend(packages_to_install)
        print("Installing missing MOI packages with:")
        print(" ".join(pip_command))
        subprocess.check_call(pip_command)
        importlib.invalidate_caches()
        missing_packages = find_missing_packages()

if missing_packages:
    requirements_file = ANALYSIS_DIR / "requirements_all_gage_notebook.txt"
    raise ImportError(
        "The current Jupyter kernel is still missing: "
        + ", ".join(missing_packages)
        + f". Build a dedicated kernel with {requirements_file}."
    )

import cvxpy as cp
installed_cvxpy_solvers = set(cp.installed_solvers())
supported_cvxpy_solvers = installed_cvxpy_solvers.intersection(
    {"OSQP", "CLARABEL", "SCS"}
)
if not supported_cvxpy_solvers:
    raise RuntimeError(
        "cvxpy is installed, but none of OSQP/CLARABEL/SCS is available. "
        f"cvxpy reports: {sorted(installed_cvxpy_solvers)}"
    )
print(f"Python executable: {sys.executable}")
print(f"cvxpy version: {cp.__version__}")
print(f"Supported installed solvers: {sorted(supported_cvxpy_solvers)}")

required_paths = [
    RUN_ROOT / "input" / "sos",
    RUN_ROOT / "input" / "swot",
    RUN_ROOT / "input" / "sword",
    RUN_ROOT / "flpe",
    SOS_RESULT_DIR,
    SOURCE_CALVAL_CSV,
]
missing_paths = [str(path) for path in required_paths if not path.exists()]
if missing_paths:
    raise FileNotFoundError("Missing required paths:\n" + "\n".join(missing_paths))

SVS_FILE = discover_svs_file(SVS_DIR)
RESULT_ROOT.mkdir(parents=True, exist_ok=True)
print(f"SVS file: {SVS_FILE}")
if os.environ.get("SLURM_JOB_ID") is None:
    warnings.warn(
        "SLURM_JOB_ID is not set. Confirm that this Jupyter kernel is on a "
        "Unity compute allocation rather than a login node before Cell 6."
    )
print("Environment and paths are ready.")

### One-time Unity environment fallback

Cell 2 normally installs a missing solver stack into the active kernel. If that kernel is read-only or compute-node network access prevents installation, create a persistent kernel once from a Unity terminal (starting in the MOI repository):

```bash
python3 -m venv /nas/cee-water/cjgleason/Yushan/Confluence_Aug/.venvs/moi-gage-correlation
source /nas/cee-water/cjgleason/Yushan/Confluence_Aug/.venvs/moi-gage-correlation/bin/activate
python -m pip install --upgrade pip
python -m pip install -r analysis/requirements_all_gage_notebook.txt
python -m ipykernel install --user --name moi-gage-correlation --display-name "Python (MOI gage correlation)"
```

Then switch this notebook to the `Python (MOI gage correlation)` kernel and run all cells again.

## Basin selection rule

The additional basins are selected using the existing **final MOI validation** results. For each basin, the notebook pools finite validation samples across available algorithms and ranks by median `|nBias|`. A candidate must have at least `MIN_ALL_GAGES` gages in the Cal/Val table and at least `MIN_VALIDATION_GAGES` independent validation gages in the SOS results. The full ranking is saved so the choice is auditable.

In [ ]:
# Cell 3 — rank gage-rich basins and select the three additional targets
from IPython.display import display

calval = read_calval_table(SOURCE_CALVAL_CSV)
validation_samples = load_moi_validation_samples(
    SOS_RESULT_DIR,
    basin_id_digits=BASIN_ID_DIGITS,
)
basin_ranking, TARGET_BASINS = build_basin_ranking(
    validation_samples,
    calval,
    FIXED_BASINS,
    n_extra_basins=N_EXTRA_BASINS,
    min_all_gages=MIN_ALL_GAGES,
    min_validation_gages=MIN_VALIDATION_GAGES,
    rank_metric=RANK_METRIC,
)
basin_ranking.to_csv(RESULT_ROOT / "basin_selection_ranking.csv", index=False)
validation_samples.to_csv(RESULT_ROOT / "source_moi_validation_samples.csv", index=False)

selection_columns = [
    "basin_id",
    "selected",
    "fixed_basin",
    "selected_extra",
    "all_gages",
    "validation_gages",
    "algorithms",
    "median_abs_nbias_moi",
    "p90_abs_nbias_moi",
]
display(basin_ranking.loc[basin_ranking["selected"], selection_columns])
print("Target basin order:", TARGET_BASINS)

In [ ]:
# Cell 4 — create the all-gage experiment CSV and locate basin input records
calval_audit = write_all_gage_calval_csv(
    SOURCE_CALVAL_CSV,
    ALL_GAGE_CALVAL_CSV,
    TARGET_BASINS,
)
calval_audit.to_csv(RESULT_ROOT / "calval_conversion_audit.csv", index=False)
display(calval_audit)
print(f"Experiment Cal/Val CSV: {ALL_GAGE_CALVAL_CSV}")

basin_catalog = discover_basin_catalog(RUN_ROOT / "input")
missing_basin_records = [basin for basin in TARGET_BASINS if basin not in basin_catalog]
if missing_basin_records:
    raise KeyError(f"Target basins missing from input JSON files: {missing_basin_records}")

input_records = []
for basin_id in TARGET_BASINS:
    record = basin_catalog[basin_id]
    input_records.append(
        {
            "basin_id": basin_id,
            "listed_reaches": len(record["reach_ids"]),
            "sos": record["sos"],
            "sword": record["sword"],
            "source_json": record["source_json"],
        }
    )
input_records = __import__("pandas").DataFrame(input_records)
input_records.to_csv(RESULT_ROOT / "selected_basin_input_records.csv", index=False)
display(input_records)

## Run MOI

Run the next cell from a Unity compute allocation. Each completed basin immediately writes its residual and solver CSV, so an interrupted notebook can resume with `REUSE_EXISTING=True`. Set `WRITE_REACH_NETCDF=True` only when full reach-level NetCDF output is needed; the residual analysis uses the exact in-memory solver diagnostics.

In [ ]:
# Cell 5 — run/resume constrained MOI for the six all-gage basins
import pandas as pd

if RUN_MOI_EXPERIMENT:
    residuals, solver_diagnostics, run_audit = run_selected_basins(
        TARGET_BASINS,
        basin_catalog,
        moi_repo=MOI_REPO,
        run_root=RUN_ROOT,
        svs_file=SVS_FILE,
        all_gage_calval_csv=ALL_GAGE_CALVAL_CSV,
        result_root=RESULT_ROOT,
        write_reach_netcdf=WRITE_REACH_NETCDF,
        verbose=VERBOSE,
        parameter_overrides=PARAMETER_OVERRIDES,
        reuse_existing=REUSE_EXISTING,
        continue_on_error=CONTINUE_ON_ERROR,
    )
else:
    residuals = pd.read_csv(
        RESULT_ROOT / "all_basins_mean_gage_residuals.csv",
        dtype={"basin_id": str, "reach_id": str},
    )
    solver_path = RESULT_ROOT / "all_basins_solver_diagnostics.csv"
    solver_diagnostics = (
        pd.read_csv(solver_path, dtype={"basin_id": str})
        if solver_path.is_file()
        else pd.DataFrame()
    )
    run_audit = pd.read_csv(RESULT_ROOT / "run_audit.csv", dtype={"basin_id": str})

display(run_audit)
if (run_audit["status"] == "failed").any():
    warnings.warn("At least one basin failed. Inspect its error_log before interpretation.")
if residuals.empty:
    raise RuntimeError("No residuals are available. Inspect run_audit.csv.")
display(
    residuals.groupby(["basin_id", "algorithm"])["reach_id"]
    .nunique()
    .rename("fitted_gages")
    .unstack(fill_value=0)
)

In [ ]:
# Cell 6 — summaries, worst-gage audit, and ECDF figure
import matplotlib.pyplot as plt

summary = summarize_residuals(residuals)
summary_path = RESULT_ROOT / "mean_gage_fit_summary.csv"
summary.to_csv(summary_path, index=False)

worst_gages = residuals.sort_values("abs_nbias", ascending=False).head(50)
worst_path = RESULT_ROOT / "worst_50_mean_gage_residuals.csv"
worst_gages.to_csv(worst_path, index=False)

display(summary.round(4))
display(
    worst_gages[[
        "basin_id", "algorithm", "reach_id", "station_id",
        "n_matched_dates", "observed_mean_cms", "fitted_mean_cms",
        "nbias", "abs_nbias"
    ]].head(30)
)

figure = plot_residual_ecdfs(
    residuals,
    TARGET_BASINS,
    signed_xlim=(-1.0, 2.0),
    absolute_xlim=(0.0, 2.0),
)
figure_path = RESULT_ROOT / "all_gage_mean_flow_residual_ecdfs.png"
figure.savefig(figure_path, dpi=300, bbox_inches="tight")
plt.show()

print(f"Summary: {summary_path}")
print(f"Worst-gage audit: {worst_path}")
print(f"ECDF figure: {figure_path}")
print(f"All outputs: {RESULT_ROOT}")

## Interpretation checklist

- A curve rising rapidly near `|nBias| = 0` means the current forward/inverse system can reproduce most gage means when all gages are supplied.
- Broad `|nBias|` ECDFs even in this in-sample experiment indicate incompatibility among gages, mass balance, topology, priors, or other forward-model constraints under the current weights.
- If all algorithms fail at the same reaches, inspect gage-to-reach mapping, date coverage, SWORD topology, and neighboring gages first.
- If only one algorithm fails, inspect that FLPE algorithm and its uncertainty/bias treatment.
- The default gage uncertainty remains a soft 10% constraint. If a basin still fits poorly, repeat under a new `RUN_TAG` with `PARAMETER_OVERRIDES = {"Gage_Uncertainty": 0.01}` to distinguish structural incompatibility from objective weighting.